In [ ]:
import os
import pandas as pd
import kagglehub

# Download/locate the dataset via kagglehub
dataset_path = kagglehub.dataset_download("dataset066/sepsis-prediction-dataset")

print("Dataset successfully loaded from cache path.")

csv_file_path = os.path.join(dataset_path, "Synthetic_ICU_Dataset_10000rows.csv")
df = pd.read_csv(csv_file_path)

print(f"Dataset shape: {df.shape}")

In [ ]:
import numpy as np

# Inspect dimensions
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# Check data types and non-null counts
print("--- Column Types & Data Summary ---")
print(df.info())


In [ ]:
# Drop the unneeded columns
columns_to_drop = ['Unnamed: 0', 'Hour', 'Unit1', 'Unit2', 'HospAdmTime', 'Patient_ID', 'ICULOS']
df_cleaned = df.drop(columns=columns_to_drop, errors='ignore')

# Separate Features (X) and Target (y)
# Target is SepsisLabel (column 42)
X_cleaned = df_cleaned.drop(columns=['SepsisLabel'])
y_cleaned = df_cleaned['SepsisLabel']

print(f"Features shape: {X_cleaned.shape}")
print(f"Target shape: {y_cleaned.shape}")

# Check remaining missing values in features
print("\n--- Remaining Missing Values in Features ---")
missing_summary = X_cleaned.isnull().sum()
print(missing_summary[missing_summary > 0])

## BASELINE

In [ ]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, roc_curve
import matplotlib.pyplot as plt

# Stratified Train-Test Split (crucial for imbalanced data to keep the 5-10% ratio in both splits)
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")
print(f"Positive sepsis cases in test set: {y_test.sum()} out of {len(y_test)}\n")

# Initialize and Train Baseline XGBoost Classifier
model = XGBClassifier(random_state=42,eval_metric='logloss')
model.fit(X_train, y_train)

# Evaluate Performance
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

# plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
plt.figure(figsize=(8, 6))
plt.plot(
    fpr, tpr, 
    color='darkorange', 
    lw=2.5, 
    label=f'Default XGBoost (AUC = {roc_auc:.4f})'
)

# Plot the diagonal random-chance line (AUC = 0.50)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guess (AUC = 0.50)')

# Formatting the plot for your portfolio
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12)
plt.title('Receiver Operating Characteristic (ROC) - Baseline', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("BASELINE RESULTS")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC (Precision-Recall AUC): {pr_auc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Since sepsis cases only make up a small percentage of cases, the model is marking all cases as not sepsis and still getting ~98% accuracy. 

# Add scale_pos_weight to penalize missed sepsis more heavily

To force XGBoost to pay attention to the positive cases, we can use the scale_pos_weight parameter. This tells the algorithm to heavily penalize false negatives (missing a sepsis patient) proportional to the class imbalance ratio.

In [ ]:
# Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2,  random_state=42, stratify=y_cleaned
)

# Calculate scale_pos_weight (Ratio of negative to positive classes in training data)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count
print(f"Calculated scale_pos_weight: {scale_weight:.2f}\n")

# Initialize and Train XGBoost with weighting
model_weighted = XGBClassifier(
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='logloss'
)
model_weighted.fit(X_train, y_train)

# Evaluate Performance
y_pred_proba = model_weighted.predict_proba(X_test)[:, 1]
y_pred = model_weighted.predict(X_test)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print("WEIGHTED RESULTS")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC (Precision-Recall AUC): {pr_auc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Introducing scale_pos_weight = 46.62 told the algorithm to penalize missing a positive sepsis case ~46 times more heavily than a false positive. However, the classification report above shows that the model **still predicted all zeros** at the default 0.5 decision threshold.

# Lower decision threshold

Because sepsis is so rare, the probabilities predicted by the model are still very low. When forced to use a rigid 0.5 threshold, anything under 50% confidence was rounded down to zero. Let's lower the decision threshold.


In [ ]:
# Look at the predicted probabilities for the actual positive cases in the test set
positive_indices = np.where(y_test == 1)[0]
positive_probs = y_pred_proba[positive_indices]

print("Predicted probabilities for actual sepsis patients in test set:")
print(positive_probs)
print(f"\nMax probability assigned to a true sepsis case: {positive_probs.max():.4f}")
print(f"Mean probability assigned to a true sepsis case: {positive_probs.mean():.4f}")

# Find decision threshold that optimizes F1-score

We can automatically find the optimal decision threshold that maximizes the F1-score for the imbalanced test set, bypassing the 0.5 default.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

# Calculate precision, recall, and thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

# Plotting the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='b', lw=2, label=f'PR-AUC = {pr_auc:.4f}')
plt.xlabel('Recall', fontsize=11)
plt.ylabel('Precision', fontsize=11)
plt.title('Precision-Recall Curve for ICU Sepsis Prediction', fontsize=13, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower left', fontsize=12)


plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

# Calculate Precision-Recall curve probabilities and thresholds
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Find the threshold that maximizes the F1-score
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Optimal Decision Threshold: {best_threshold:.5f}")
print(f"Best F1-Score at this threshold: {best_f1:.4f}\n")

# Apply the new threshold to generate predictions
y_pred_tuned = (y_pred_proba >= best_threshold).astype(int)

# Print the new classification report
print("Maximized F1 Threshold Classification Report")
print(classification_report(y_test, y_pred_tuned, zero_division=0))

# Re-evaluate PR-AUC
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba):.4f}")

Dropping the threshold to 0.00232 allowed us to catch 38% of actual sepsis cases (**Recall = 0.38**) in the test set.
**Precision sits at 0.03 (3%)**, meaning that for every true sepsis case caught, the model generates a high volume of false alarms. 
 
The compressed probability distribution indicates that our shallow default trees lack the capacity to build strong internal confidence. This justifies moving on to Hyperparameter Tuning (adjusting tree depth and learning rate) to expand our probability range and improve overall predictive power.

## Increase tree depth and adjust learning rate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

# Initialize and Train Tuned XGBoost Classifier
deeper_model = XGBClassifier(
    scale_pos_weight=scale_weight,
    max_depth=5,              # deeper to learn more complex interactions
    learning_rate=0.04,       
    n_estimators=200,         # More trees to refine predictions
    subsample=0.7,            # Row subsampling to reduce overfitting
    colsample_bytree=0.7,     # Feature subsampling per tree
    random_state=42,
    eval_metric='logloss'
)

deeper_model.fit(X_train, y_train)

# Evaluate Probabilities
y_pred_proba_tuned = deeper_model.predict_proba(X_test)[:, 1]

positive_indices = np.where(y_test == 1)[0]
positive_probs = y_pred_proba_tuned[positive_indices]

print(f"--- Tuned Probabilities for True Sepsis Cases ---")
print(f"Max probability assigned to a true sepsis case: {positive_probs.max():.4f}")
print(f"Mean probability assigned to a true sepsis case: {positive_probs.mean():.4f}\n")

# Find Optimal Threshold via PR Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_tuned)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred_tuned_thresh = (y_pred_proba_tuned >= best_threshold).astype(int)

# Print Metrics
print(f"Optimal Threshold: {best_threshold:.5f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba_tuned):.4f}\n")
print("--- Tuned Classification Report ---")
print(classification_report(y_test, y_pred_tuned_thresh, zero_division=0))

## Try missingness indicators

In clinical data, missing values are almost never "missing at random." The fact that a test was not ordered can be just as clinically informative as the test result itself. Routine vitals (heart rate, blood pressure) are monitored continuously, but expensive or invasive lab panels (like arterial blood gases, lactate, cardiac enzymes, or specific cultures) are ordered reactively. Let's see if we can improve results with missingness indicators.

In [ ]:
print(X_cleaned.info())

In [ ]:
# Select key intermittent lab and blood gas features for missingness flags
df_features = df_cleaned.copy()

features_to_flag = [
    'Lactate', 'Creatinine', 'BUN', 'AST', 'Alkalinephos', 
    'Bilirubin_total', 'pH', 'PaCO2', 'BaseExcess', 'HCO3', 
    'Platelets', 'PTT', 'Fibrinogen', 'WBC', 'TroponinI'
]

# Create binary missingness columns (1 if missing, 0 if present)
for col in features_to_flag:
    if col in df_features.columns:
        df_features[f'{col}_missing'] = df_features[col].isna().astype(int)

print(f"Added {len(features_to_flag)} missingness indicator features.")

X_indicators = df_features.drop(columns=['SepsisLabel'])
y_indicators = df_features['SepsisLabel']

print(f"Features shape: {X_indicators.shape}")
print(f"Target shape: {y_indicators.shape}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_indicators, y_indicators, test_size=0.2, random_state=42, stratify=y_indicators
)

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_weight = neg_count / pos_count

# Initialize and Train Tuned XGBoost Classifier
indicators_model = XGBClassifier(
    scale_pos_weight=scale_weight,
    max_depth=5,              # Allow trees to learn more complex interactions
    learning_rate=0.04,       # Slower learning rate for better convergence
    n_estimators=200,         # More trees to refine predictions
    subsample=0.7,            # Row subsampling to reduce overfitting
    colsample_bytree=0.7,     # Feature subsampling per tree
    random_state=42,
    eval_metric='logloss'
)

indicators_model.fit(X_train, y_train)

# Evaluate Probabilities
y_pred_proba_tuned = indicators_model.predict_proba(X_test)[:, 1]

positive_indices = np.where(y_test == 1)[0]
positive_probs = y_pred_proba_tuned[positive_indices]

print(f"Tuned Probabilities for True Sepsis Cases (INDICATORS ADDED)")
print(f"Max probability assigned to a true sepsis case: {positive_probs.max():.4f}")
print(f"Mean probability assigned to a true sepsis case: {positive_probs.mean():.4f}\n")

# Find Optimal Threshold via PR Curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba_tuned)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_pred_tuned_thresh = (y_pred_proba_tuned >= best_threshold).astype(int)

# Print Metrics
print(f"Optimal Threshold: {best_threshold:.5f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_tuned):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, y_pred_proba_tuned):.4f}\n")
print("Tuned Classification Report (INDICATORS ADDED)")
print(classification_report(y_test, y_pred_tuned_thresh, zero_division=0))

That didn't really make much of a difference. Let's go back to the version without the added indicators.

# SHAP

Let's analyze our tuned model

In [ ]:
import shap

X_train, X_test, y_train, y_test = train_test_split(
    X_cleaned, y_cleaned, test_size=0.2, random_state=42, stratify=y_cleaned
)


# Initialize the Tree Explainer on our best-performing tuned model
explainer = shap.TreeExplainer(deeper_model)

# Calculate SHAP values for the test set
shap_values = explainer(X_test)

# Global Feature Importance Bar Plot
# This displays the mean absolute SHAP value for each feature across the dataset
print("--- Global Feature Importance (Bar Plot) ---")
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Detailed Bee Swarm Summary Plot
# This shows both feature importance AND the direction of impact (high vs. low feature values)
print("--- SHAP Bee Swarm Plot (Impact & Direction) ---")
shap.summary_plot(shap_values, X_test)